# Subsetting PACE OCI Data

imported on: **2026-11-16**

<p>This notebook is from NASA's OB.DAAC Help Hub (oceandata-notebooks).</p>

> The original source for this document is [https://github.com/nasa/oceandata-notebooks](https://github.com/nasa/oceandata-notebooks)

# Subsetting PACE OCI data

**Authors:** Anna Windle (NASA, SSAI), Carina Poulin (NASA, SSAI), Ian Carroll (NASA, UMBC) 

Last Updated: July 7, 2026

<div class="alert alert-success" role="alert">

The following notebooks are **prerequisites** for this tutorial.

- Learn with OCI: [Data Access](oci_data_access)

</div>

<div class="alert alert-info" role="alert">

An [Earthdata Login][edl] account is required to access data from the NASA Earthdata system, including NASA ocean color data.

</div>

[edl]: https://urs.earthdata.nasa.gov/

## Summary

This tutorial demonstrates how to subset PACE OCI L2 and L3M data. L2 data is subsetted here using `harmony-py`, while L3M data is subsetted using `xarray`. 

[Harmony] is a web service that allows you to customize many NASA datasets, including the ability to subset, reproject, and reformat files. Data can be subsetted for a geographic region, a temporal range and by variable. In some cases, data can be “reprojected” from its native coordinate reference system (CRS) to the coordinate reference system relevant to your analysis. Data can also be reformatted from its native file format to a format that is more relevant for your application. These services are collectively called transformation services. However, not all services and transformations are available for all datasets. You will learn how to discover which services are available for a given dataset.

Harmony services can be used in multiple ways:
1. through a graphical user interface (GUI) while downloading applicable granules from [Earthdata Search],
2. by direct requests to [Harmony's RESTful API],
3. through [SeaDAS] ≥ 11.0.0,
4. or, as in this tutorial, using the `harmony-py` Python package.

The Python package handles NASA Earthdata Login (EDL) authentication and optionally integrates with the CMR Python Wrapper by accepting collection results as a request parameter. It's convenient for scientists who wish to use Harmony from Jupyter notebooks.
After this tutorial, you can dive deeper into `harmony-py` on [ReadTheDocs](https://harmony-py.readthedocs.io/en/main/). 

This tutorial demonstrates how to subset and reformat PACE OCI data from the NASA Earthdata Cloud using `harmony-py`. It is modelled off of tutorials developed by [NSIDC] and [NASA-Openscapes]. 

[Harmony]: https://harmony.earthdata.nasa.gov/
[Earthdata Search]: https://search.earthdata.nasa.gov/
[Harmony's RESTful API]: https://harmony.earthdata.nasa.gov/docs
[harmony-py]: https://github.com/nasa/harmony-py
[NSIDC]: https://github.com/nsidc/NSIDC-Data-Tutorials/blob/main/notebooks/NASA_Earthdata_webinar_short/harmony-py-webinar-short.ipynb
[NASA-Openscapes]: https://nasa-openscapes.github.io/earthdata-cloud-cookbook/tutorials/Harmony.html
[SeaDAS]: https://www.earthdata.nasa.gov/data/tools/seadas

## Learning Objectives

At the end of this notebook you will know how to:

- Use the `harmony-py` Python library to spatially and temporally subset PACE OCI Level-2 data
- Download the subsetted data to your local directory
- Stream the subsetted data
- Open and plot subsetted L2 PACE OCI data
- Open and subset PACE OCI Level-3 mapped data using `xarray`

## 1. Setup

Begin by importing all of the packages used in this notebook.

In [ ]:
from datetime import datetime
from pathlib import Path

import earthaccess
import harmony
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rioxarray
import xarray as xr
from rasterio.enums import Resampling

## 2. Earthdata authentication and Harmony client initalization

The `earthaccess` package lets us easily authenticate with NASA's Earthdata Login (EDL) and pass an EDL token to Harmony.

In [ ]:
auth = earthaccess.login()
harmony_client = harmony.Client(token=earthaccess.get_edl_token())

## 3. Discover subsetting capabilities for Level-2 PACE OCI data

Define which data set you’d like to access using either the dataset `short_name` or its collection `concept_id`. For this example, we’ll use PACE OCI Level-2 Regional Ocean Biogeochemical Properties Data (PACE_OCI_L2_BGC).

In [ ]:
capabilities_request = harmony.CapabilitiesRequest(short_name="PACE_OCI_L2_BGC")
capabilities = harmony_client.submit(capabilities_request)
capabilities["summary"]

You can see here under `['summary']` that there is a Harmony service for spatial subsetting by a bounding box or shape.

## 4. Build and submit a request

The Harmony service is based around user-submitted jobs. Using the service is a two-step process of (1) creating a specific type of request, and (2) passing the request to the `harmony_client` in exchange for some response from Harmony.

One type of request, the `JobsRequest`, let's you check for any previously submitted jobs. You can filter these jobs using labels, which we will learn how to apply below.

First step, create a request:

In [ ]:
request = harmony.JobsRequest(labels=["help-hub-tutorial"])

Second step, get a response to your request:

In [ ]:
response = harmony_client.submit(request)

For your first time through this tutorial, you aren't expecting to recover any previously submitted jobs, so `response["count"]` ought to be zero. If you've already submitted a job with this label (e.g. because you are re-running this tutorial), then the response includes information about that existing job.

Let's continue to build a request for the subsetting job we want the service to run. Using the "labels" keyword tags jobs so that we can easily find and re-use results from this job later.

In [ ]:
request = harmony.Request(
    collection=harmony.Collection(id="PACE_OCI_L2_BGC"),
    spatial=harmony.BBox(-76.75, 36.97, -75.74, 39.01),
    temporal={"start": datetime(2025, 7, 1), "stop": datetime(2025, 8, 1)},
    variables=["geophysical_data/chlor_a", "geophysical_data/l2_flags"],
    pixel_subset=True,
    labels=request.labels,
)

Depending on whether we got a non-zero count from the `JobsRequest`, we can either submit the subsetting request or dig into the job info we retrieved. Either way, we come up with a `job_id` that uniquely identifies the subsetting job.

In [ ]:
if response["count"]:
    job_id = response["jobs"][0]["jobID"]
else:
    job_id = harmony_client.submit(request)
job_id

The `job_id` can be used to check on the details of your job progress. If you are logged into Earthdata, you can see your jobs:

> https://harmony.earthdata.nasa.gov/workflow-ui

Or watch your progress here by calling the `wait_for_processing` method:

In [ ]:
harmony_client.wait_for_processing(job_id, show_progress=True)

**Note**:
Requesting a job can take a variable amount of time. In our experience, running this particular job has ranged from 2 to 24 minutes. If you run into trouble and need to replace this job, remove it's label and start again from the top:
```python
harmony_client.submit(harmony.DeleteLabelsRequest(labels=request.labels, job_ids=[job_id]))
```

In [ ]:
job_summary = harmony_client.result_json(job_id)

In [ ]:
print("Number of granules:", job_summary["numInputGranules"])
print("Original Data Size:", job_summary["originalDataSize"])
print("Output Data Size:", job_summary["outputDataSize"])
print("Data Size % Change:", job_summary["dataSizePercentChange"])

If you want to access a specific job that has already run, you can simply assign a known id to `job_id` and continue below.

Results are kept for 30 days in the Harmony S3 bucket.

## 5. Access the subsetted data

The subsetted files can be accessed by downloading the files to a local machine, or by streaming the data. We will use both access methods in the examples below.

### Download a single file

The download method takes a url to a single subsetted file. There are two optional arguments; `directory` used to specify an *existing* folder for storing data (defaults to the current directory), and `overwrite` which defaults to `False` to avoid downloading the same file twice. If you need to download the file again, then set `overwrite=True`.

Let's download the first granule:

In [ ]:
url = list(harmony_client.result_urls(job_id))[0]
filepath = harmony_client.download(url).result()

You should see this file saved in your local directory.

You can also make a new folder in your local directory to save the subsetted data. Here, we are naming it "subsetted_data":

In [ ]:
subsetted_data = Path("subsetted_data")
subsetted_data.mkdir(exist_ok=True)

filepath = harmony_client.download(url, directory=subsetted_data).result()

### Download all files

The `download_all` method can use the `job_id` or the `result_json`, which contains result URLs.

As with `download`, the download directory path on the local machine can be specified with the `directory` keyword. To save downloading the same file, the `overwrite` keyword can be set to `False`.

The paths fo the files are returned as a list.

In [ ]:
futures = harmony_client.download_all(job_id, directory=subsetted_data)
filelist = [f.result() for f in futures]

In [ ]:
len(filelist)

You can now open the files using `xarray`.

In [ ]:
dt = xr.open_datatree(filelist[0])
ds = xr.merge(dt.to_dict().values())
ds = ds.set_coords(("longitude", "latitude"))
ds

There's no need to keep these files around if you plan to stream the data instead of downloading. You can run this to delete all of the locally downloaded files.

In [ ]:
for item in subsetted_data.glob("*.nc*"):
    item.unlink()

### Stream the files

If you are working in the AWS `us-west-2` region (the same region as NASA Earthdata Cloud) you can *stream* the data using direct S3 access.

You must be running this notebook in the AWS us-west-2 region to run the following code cells.

We need to get the url for the data in the S3 bucket. We can do this using `result_urls`, as we did for `download` but we set `link_type=LinkType.s3` to specify we want the S3 url.

In [ ]:
urls = list(harmony_client.result_urls(job_id, link_type=harmony.LinkType.s3))
urls

We need AWS credentials to access the S3 bucket with the results. These can be passed when using `xr.open_datatree`:

In [ ]:
kwargs = {
    "engine": "h5netcdf",
    "storage_options": {
        "client_kwargs": harmony_client.aws_credentials(),
    },
}
dt = xr.open_datatree(urls[0], **kwargs)
ds = xr.merge(dt.to_dict().values())
ds = ds.set_coords(("longitude", "latitude"))
ds

## 6. Plot the subsetted data

Let's do a quick plot of `chlor_a` from the first granule:

In [ ]:
plot = ds["chlor_a"].plot(x="longitude", y="latitude")

Now let's plot multiple subsetted granules:

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4), constrained_layout=True)
axes = axes.ravel()
da = []

for ax, file in zip(axes, urls[:10]):

    dt = xr.open_datatree(file, **kwargs)
    ds = xr.merge(dt.to_dict().values())
    ds = ds.set_coords(("longitude", "latitude"))

    da.append(ds["chlor_a"])

    date = ds.attrs["time_coverage_start"]
    im = ds["chlor_a"].plot(
        ax=ax,
        x="longitude",
        y="latitude",
        cmap="viridis",
        add_colorbar=False,
        vmin=0,
        vmax=20,
    )
    ax.set_title(date, fontsize=8)

    ax.set_xlabel("")
    ax.set_ylabel("")

fig.colorbar(im, ax=axes, orientation="vertical", shrink=0.8, label="Chl a (mg m-3)")
plt.show()

We can also make a L2 composite. However, in order to combine multiple granules into a common spatial framework we need to project the data onto a defined grid with a given reslution. We will use code presented in the [Projecting PACE Data onto a Predefined Grid tutorial.](https://nasa.github.io/oceandata-notebooks/notebooks/oci/oci_grid_match.html)

In [ ]:
def grid_data(src, resolution, dst_crs="epsg:4326", resampling=Resampling.nearest):
    """
    Reproject a L2 dataset to match an input grid. Makes sure 3D variables are
        in (Z, Y, X) dimension order, and all variables have spatial dims/crs 
        assigned.
    Args:
        src - an xarray dataset or dataarray to reproject
        resolution - resolution of the output grid, in dst_crs units
        dst_crs - CRS of the output data
        resampling - resampling method (see rasterio.enums)
    Returns:
        dst - projected xr dataset
    """
    if (len(list(src.dims)) == 3) and (list(src.dims)[0] != "wavelength"):
        src = src.transpose("wavelength", ...)
    src = src.rio.set_spatial_dims("pixels_per_line", "number_of_lines")
    src = src.rio.write_crs("epsg:4326")

    # Calculating the default affine transform
    defaults = rasterio.warp.calculate_default_transform(
        src.rio.crs,
        dst_crs,
        src.rio.width,
        src.rio.height,
        left=src.attrs["geospatial_lon_min"],
        bottom=src.attrs["geospatial_lat_min"],
        right=src.attrs["geospatial_lon_max"],
        top=src.attrs["geospatial_lat_max"],
    )
    # Aligning that transform to our desired resolution
    transform, width, height = rasterio.warp.aligned_target(*defaults, resolution)
    
    dst = src.rio.reproject(
        dst_crs=dst_crs,
        shape=(height, width),
        transform=transform,
        src_geoloc_array=(
            src["longitude"],
            src["latitude"],
        ),
        nodata=np.nan,
        resample=resampling,
    )
    dst["x"] = dst["x"].round(9)
    dst["y"] = dst["y"].round(9)
    
    return dst.rename({"x":"longitude", "y":"latitude"})

Now, we can make a 10-day Chl a composite. The `l2_flags` variable cannot be gridded using this function; if you need to use the `l2_flags` to mask out additional data, you can do that on the subetted granules before gridding.

In [ ]:
gridded_list = []
resolution = (0.015, 0.015)

for file in urls[:10]:

    dt = xr.open_datatree(file, **kwargs)
    ds = xr.merge(dt.to_dict().values())
    ds = ds.set_coords(("longitude", "latitude"))
    ds = ds.drop_vars("l2_flags")

    ds_gridded = grid_data(ds, resolution)

    gridded_list.append(ds_gridded.chlor_a)

stack = xr.concat(gridded_list, dim="scene")

chlor_a_mean = stack.mean(dim="scene", skipna=True)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

chlor_a_mean.plot(
    ax=ax,
    cmap="viridis",
    vmin=0,
    vmax=20
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

plt.tight_layout()
plt.show()

## 7. Subsetting L3M data

Currently, `harmony-py` does not support spatial subsetting for PACE OCI L3M products. You can verify this by submitting a Harmony request and inspecting the available services, where the subset capabilities are listed as `False`.

In [ ]:
capabilities_request = harmony.CapabilitiesRequest(short_name="PACE_OCI_L3M_BGC")
capabilities = harmony_client.submit(capabilities_request)
capabilities["summary"]

Because spatial subsetting is not currently supported for PACE OCI L3M products through Harmony, we will instead use `xarray` to subset the data using spatial coordinates. Since L3M products are already mapped to a regular spatial grid, this approach is both straightforward and computationally efficient.

Let's begin by opening a monthly (MO) PACE_OCI_L3M_BGC composite at 4 km spatial resolution using `earthaccess`:

In [ ]:
results = earthaccess.search_data(
    short_name="PACE_OCI_L3M_BGC",
    temporal=("2025-07", "2025-07"),
    granule_name="*.MO.*.4km.*",
)
paths = earthaccess.open(results)

In [ ]:
ds = xr.open_dataset(paths[0])
ds

Now we are going to subset in two ways, first, by choosing one variable with `["chlor_a"]`, and secondly by selecting a spatial subset with `slice`. We can do this all in one line:

In [ ]:
ds_sub = ds["chlor_a"].sel({"lat": slice(39.01, 36.97), "lon": slice(-76.75, -75.74)})
plot = ds_sub.plot.imshow()

If we now save this sliced dataset to a netCDF file, we have effectively "downloaded" a subsetted dataset:

In [ ]:
subsetted_data = Path("./subsetted_data")
subsetted_data.mkdir(exist_ok=True)
path = subsetted_data / ds.attrs["product_name"]
path = path.with_suffix(".subsetted.nc")
print(path)

In [ ]:
ds_sub.to_netcdf(path)

You should now see this file in your "subsetted_data" folder. Let's compare the size to the original L3M file:

In [ ]:
og_size = results[0].size
sub_size = path.stat().st_size / 2**20
print(f"Original Data Size: {og_size:.2f} MB")
print(f"Output Data Size: {sub_size:.2f} MB")
print(f"Data Size % Change: {100 * (1 - sub_size / og_size):.2f} % reduction")

<div class="alert alert-info" role="alert">

You have completed the notebook on Subsetting PACE OCI data!

</div>